# M06.P01 — Runnables, prompts y structured output


In [ ]:
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import PydanticOutputParser

class IncidentAnalysis(BaseModel):
    incident_id: str
    category: str
    requires_review: bool

parser = PydanticOutputParser(pydantic_object=IncidentAnalysis)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Analiza el incidente y devuelve únicamente el formato solicitado.\n{format_instructions}"),
    ("user", "ID: {incident_id}\nServicio: {service}\nResumen: {summary}")
]).partial(format_instructions=parser.get_format_instructions())

data = {
    "incident_id": "INC-2048",
    "service": "payments-api",
    "summary": "Degradación causada por una configuración incorrecta del balanceador.",
}

prompt_value = prompt.invoke(data)
print(type(prompt_value))
print(prompt_value.to_string())


In [ ]:
def deterministic_demo(prompt_value):
    # TODO: devuelve JSON compatible con IncidentAnalysis.
    raise NotImplementedError

mock_model = RunnableLambda(deterministic_demo)
chain = prompt | mock_model | parser
# TODO: chain.invoke(data)


## Batch y RunnableParallel


In [ ]:
cases = [
    data,
    {"incident_id":"INC-2051","service":"customer-portal","summary":"Errores intermitentes de autenticación."},
]
# TODO: chain.batch(cases)
# TODO: RunnableParallel con analysis, summary_length y mentions_load_balancer.


## GPT-5.6 Luna en Amazon Bedrock

El Space está en `us-east-1`. Las credenciales proceden del SageMaker Execution Role. No incluyas access keys en el notebook.


In [ ]:
from langchain_aws import ChatBedrockConverse

MODEL_ID = "us.openai.gpt-5.6-luna"
REGION = "us-east-1"
model = ChatBedrockConverse(model=MODEL_ID, region_name=REGION, temperature=0)

# TODO:
# structured_model = model.with_structured_output(IncidentAnalysis)
# real_chain = prompt | structured_model
# result = real_chain.invoke(data)
# comprueba isinstance(result, IncidentAnalysis)
